In [ ]:
"""
Vietnamese News Crawler - Fixed Version
Crawls articles from Báo Mới, Thanh Niên, VNExpress, Dân Trí
"""

import requests
from bs4 import BeautifulSoup
import time
import os
import re
from datetime import datetime
from typing import List, Dict, Tuple
from urllib.parse import urljoin

# Configuration
HEADERS = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36'
}

OUTPUT_DIR = 'raw_data'
os.makedirs(OUTPUT_DIR, exist_ok=True)

MAX_SENTENCES = 1000000  # Maximum number of sentences to collect

# ==================== Text Processing ====================
def clean_text(text: str) -> str:
    """Clean and normalize Vietnamese text"""
    if not text:
        return ""
    
    # Remove extra whitespace
    text = re.sub(r'\s+', ' ', text).strip()
    # Remove special characters but keep Vietnamese diacritics
    text = re.sub(r'[\n\r\t]', ' ', text)
    
    return text

def split_into_sentences(text: str) -> List[str]:
    """Split Vietnamese text into sentences"""
    if not text:
        return []
    
    # Vietnamese sentence endings
    text = text.replace('.\n', '.')
    text = text.replace('!\n', '!')
    text = text.replace('?\n', '?')
    
    # Split by sentence endings: . ! ? and also by em dash (–)
    sentences = re.split(r'(?<=[.!?–\-])\s+(?=[A-ZÀỂẦẬĂẰẲẴẶẢẠẪẨễ])', text)
    
    # If no period-based split worked, split by longer text blocks
    if len(sentences) < 2:
        sentences = re.split(r'(?<=[.!?])\s+', text)
    
    # If still just one sentence, split by newlines as last resort
    if len(sentences) < 2:
        sentences = text.split('\n')
    
    # Clean each sentence - require at least 5 characters to be valid
    sentences = [clean_text(s) for s in sentences if len(clean_text(s)) > 5]
    
    return sentences

# ==================== VNExpress Crawler (với Pagination) ====================
def crawl_vnexpress(num_articles: int = 50, max_pages: int = 20, max_sentences: int = None) -> Tuple[List[str], Dict[str, int]]:
    """Crawl articles from VNExpress with pagination support
    
    VNExpress pagination pattern:
    - Page 1: https://vnexpress.net/thoi-su
    - Page 2: https://vnexpress.net/thoi-su-p2
    - Page 3: https://vnexpress.net/thoi-su-p3
    
    Returns:
    - List of sentences
    - Dictionary with category stats
    """
    if max_sentences is None:
        max_sentences = MAX_SENTENCES
    
    sentences = []
    urls_visited = set()
    category_stats = {}  # Track sentences per category
    
    print("Crawling VNExpress (with pagination)...")
    
    # Only use top categories for pagination
    base_urls = [
        'https://vnexpress.net/thoi-su',
        'https://vnexpress.net/the-gioi',
        'https://vnexpress.net/kinh-doanh',
        'https://vnexpress.net/khoa-hoc-cong-nghe',
        'https://vnexpress.net/goc-nhin',
        'https://vnexpress.net/spotlight',
        'https://vnexpress.net/bat-dong-san',
        'https://vnexpress.net/suc-khoe',
        'https://vnexpress.net/giai-tri',
        'https://vnexpress.net/the-thao',
        'https://vnexpress.net/phap-luat',
        'https://vnexpress.net/giao-duc',
        'https://vnexpress.net/doi-song',
        'https://vnexpress.net/oto-xe-may',
        'https://vnexpress.net/du-lich',
        'https://vnexpress.net/anh',
        'https://vnexpress.net/infographics',
        'https://vnexpress.net/y-kien',
        'https://vnexpress.net/tam-su',
        'https://vnexpress.net/thu-gian',
    ]
    
    for base_url in base_urls:
        if len(urls_visited) >= num_articles or len(sentences) >= max_sentences:
            break
        
        category_name = base_url.split('/')[-1]
        category_stats[base_url] = 0  # Initialize category counter
        print(f"\n  Category: {category_name}")
        
        # Crawl multiple pages for each category
        for page in range(1, max_pages + 1):
            if len(urls_visited) >= num_articles or len(sentences) >= max_sentences:
                break
            
            # Construct page URL
            if page == 1:
                page_url = base_url
            else:
                page_url = f"{base_url}-p{page}"
            
            try:
                print(f"    Page {page}: ", end="")
                response = requests.get(page_url, headers=HEADERS, timeout=15)
                response.encoding = 'utf-8'
                soup = BeautifulSoup(response.content, 'html.parser')
                
                # Find all .html links on this page
                article_links = soup.find_all('a', href=re.compile(r'\.html$', re.I))
                
                if not article_links:
                    print("No articles (stopping pagination)")
                    break
                
                print(f"Found {len(article_links)} articles")
                articles_on_page = 0
                
                for links in article_links[:30]:
                    if len(urls_visited) >= num_articles or len(sentences) >= max_sentences:
                        break
                    
                    try:
                        article_url = links.get('href')
                        if not article_url or article_url in urls_visited:
                            continue
                        
                        article_url = urljoin(page_url, article_url)
                        
                        if 'vnexpress.net' not in article_url:
                            continue
                        
                        urls_visited.add(article_url)
                        
                        # Fetch and parse article
                        art_response = requests.get(article_url, headers=HEADERS, timeout=15)
                        art_soup = BeautifulSoup(art_response.content, 'html.parser')
                        
                        # Extract paragraphs
                        paragraphs = art_soup.find_all('p')
                        
                        if paragraphs:
                            text = ' '.join([p.get_text() for p in paragraphs])
                            text = clean_text(text)
                            
                            if len(text) > 100:
                                article_sentences = split_into_sentences(text)
                                if article_sentences:
                                    # Only add sentences if we haven't reached the max
                                    remaining = max_sentences - len(sentences)
                                    if remaining > 0:
                                        sentences_to_add = article_sentences[:remaining]
                                        sentences.extend(sentences_to_add)
                                        category_stats[base_url] += len(sentences_to_add)
                                        articles_on_page += 1
                                        title = links.get_text()[:45].strip()
                                        print(f"        ✓ [{len(sentences_to_add)} sents] {title}...")
                        
                        time.sleep(0.3)
                    
                    except Exception as e:
                        continue
                
            except Exception as e:
                print(f"Error: {str(e)[:50]}")
                continue
        
        # Print category summary
        if category_stats[base_url] > 0:
            print(f"    → {category_name}: {category_stats[base_url]} sentences")
    
    print(f"\nVNExpress Total: {len(sentences)} sentences\n")
    return sentences, category_stats


# ==================== Main Function ====================
def crawl_all_news_sources(articles_per_source: int = 10000, output_file: str = None, vnexpress_pages: int = 20, dan_tri_pages: int = 20, max_sentences: int = MAX_SENTENCES):
    print("=" * 70)
    print("Vietnamese News Crawler with BeautifulSoup - WITH PAGINATION")
    print("=" * 70)
    print(f"Start: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print(f"VNExpress pages to crawl: {vnexpress_pages}")
    print(f"Dân Trí pages to crawl: {dan_tri_pages}")
    print(f"Maximum sentences limit: {max_sentences}\n")
    
    all_sentences = []
    all_stats = {}
    
    # Crawl VNExpress
    vnexpress_sentences, vnexpress_stats = crawl_vnexpress(
        articles_per_source, 
        max_pages=vnexpress_pages, 
        max_sentences=max_sentences
    )
    all_sentences.extend(vnexpress_sentences)
    all_stats['VNExpress'] = vnexpress_stats
    
    # Remove duplicates
    unique_sentences = list(dict.fromkeys(all_sentences))
    
    # Save results
    if output_file is None:
        timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
        output_file = f"{OUTPUT_DIR}/crawled_news_{timestamp}.txt"
    else:
        output_file = os.path.join(OUTPUT_DIR, output_file)
    
    with open(output_file, 'w', encoding='utf-8') as f:
        for sentence in unique_sentences:
            f.write(sentence + '\n')
    
    # Detailed Summary
    print("=" * 70)
    print("CRAWLING COMPLETE - DETAILED STATISTICS")
    print("=" * 70)
    
    # # VNExpress statistics
    print("\n📰 VNExpress Statistics:")
    print("-" * 70)
    vnexpress_total = 0
    for url, count in vnexpress_stats.items():
        if count > 0:
            category = url.split('/')[-1]
            print(f"  {category:30s}: {count:6d} sentences")
            vnexpress_total += count
    print(f"  {'VNExpress Total':30s}: {vnexpress_total:6d} sentences")
    
    # Overall statistics
    print("\n" + "=" * 70)
    print("OVERALL STATISTICS:")
    print("=" * 70)
    print(f"  Total from all sources:       {len(all_sentences):6d} sentences")
    print(f"  Unique sentences:             {len(unique_sentences):6d} sentences")
    print(f"  Duplicates removed:           {len(all_sentences) - len(unique_sentences):6d} sentences")
    print(f"\n✓ Saved to: {output_file}")
    print(f"End: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print("=" * 70)


In [ ]:
if __name__ == '__main__':
    # Test with fewer pages to debug
    crawl_all_news_sources(
        articles_per_source=5000, 
        vnexpress_pages=20, 
        dan_tri_pages=20, 
        output_file='test_crawl.txt'
    )